# model/02 — All ANTHEIA Models + Ablations

Trains and evaluates all five primary ANTHEIA models and two pollinator-side
ablation models on a single shared pair universe and train/test split.

---

## Primary Models

All use the same plant-side spatial embedding Vf and pollinator-side
spatial embedding Vp. They differ only in what temporal or probabilistic
signal is added.

| Model | Feature Vector | Dim | Description |
|---|---|---|---|
| Spatial Baseline | Vf + Vp + N | 31D | No temporal signal |
| ANTHEIA-Scalar | Vf + Vp + N + Δ | 32D | PPE temporal overlap scalar |
| ANTHEIA-4D | Vf + Vp + N + V_δ (4D) | 35D | PPE spatiotemporal embedding |
| ANTHEIA-15D | Vf + Vp + N + V_δ (15D) | 46D | PPE spatiotemporal embedding |
| ANTHEIA-PMf | Vf_prob + Vp + N | 31D | PPE probability replaces binary Vf |

---

## Pollinator-Side Ablation Models

These models test whether GBIF-derived temporal information on the
**pollinator side** improves performance. Both are negative results.

| Model | Feature Vector | Dim | Description |
|---|---|---|---|
| ANTHEIA-PMp | Vf + PMp + N | 31D | Normalized GBIF weekly histogram replaces binary Vp |
| ANTHEIA-TMp | Vf + TMp + N | 31D | Binary temporal existence matrix replaces binary Vp |

**Why they fail:** GBIF pollinator observations are opportunistic citizen
science records biased by sampling effort. Weekly observation histograms
reflect when observers were active outdoors, not when pollinators were
active. Adding this biased temporal signal degrades performance — the model
learns observation-density patterns rather than ecological ones.

**Asymmetry finding:** Plant-side temporal enrichment (PPE) consistently
helps. Pollinator-side temporal enrichment (GBIF) consistently hurts.
PPE works because it is climate-driven and model-predicted, independent
of observation effort.

---

## Motivation for SDM

The pollinator-side ablation failure motivates the SDM comparison
(see `evaluation/02_sdm_comparison.ipynb`). If GBIF-derived temporal
signal hurts because it is observation-biased, then **model-predicted**
pollinator activity curves (SDM, Dan Cher) — independent of observation
effort — should improve performance. SDM is the pollinator-side analog
of what PPE provides on the plant side.

The SDM variant models use the same five architectures above but replace
GBIF a_curves with SDM-derived curves:

| SDM Variant | Replaces |
|---|---|
| Spatial Baseline-SDM | Spatial Baseline |
| ANTHEIA-Scalar-SDM | ANTHEIA-Scalar |
| ANTHEIA-4D-SDM | ANTHEIA-4D |
| ANTHEIA-15D-SDM | ANTHEIA-15D |
| ANTHEIA-PMf-SDM | ANTHEIA-PMf |

Full SDM results (25,466-species coverage) are pending Dan's complete
species distribution model build.

---

**Single seed run (seed 42).** See `evaluation/01_seed_testing.ipynb`
for 5-seed results on the primary models.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path
import gc

BASE     = Path("/scratch/ariana.l")
OLD_S4   = BASE / "Stage 4 Link Prediction Model"
NEW_S4   = BASE / "New Stage 4 Link Prediction Model"
STAGE5   = BASE / "Stage 5 PPE Representation Study"

VF_PATH      = OLD_S4 / "stage4_Vf_phenofield.csv"
F_PATH       = OLD_S4 / "stage4_F_existence_phenofield.csv"
VP_PATH      = NEW_S4 / "stage4_Vp_corrected.csv"
P_PATH       = NEW_S4 / "stage4_P_existence_corrected.csv"
GLOBI_PATH   = OLD_S4 / "stage4_globi_conus_broad.csv"
F_CURVES_PATH = NEW_S4 / "f_curves_ppe.csv"
A_CURVES_PATH = NEW_S4 / "a_curves_corrected.csv"
VDELTA_4D    = STAGE5 / "stage5_Vdelta_ppe.csv"
VDELTA_15D   = STAGE5 / "stage5_Vdelta_15d.csv"
VF_PROB      = STAGE5 / "stage5_Vf_prob.csv"
VP_PROB      = STAGE5 / "stage5_Vp_prob.csv"   # PMp — ANTHEIA-PMp ablation
VP_TEMP      = STAGE5 / "stage5_Vp_temp.csv"   # TMp — ANTHEIA-TMp ablation

SEED      = 42
NEG_RATIO = 3

print("Paths OK")

In [ ]:
# Load all embeddings
print("Loading plant side...")
Vf_df  = pd.read_csv(VF_PATH,  index_col=0)
F_df   = pd.read_csv(F_PATH,   index_col=0)
Vd4_df = pd.read_csv(VDELTA_4D,  index_col=0)
Vd15_df = pd.read_csv(VDELTA_15D, index_col=0)
Vfp_df = pd.read_csv(VF_PROB,  index_col=0)
print(f"  Vf   : {Vf_df.shape}")
print(f"  Vd4  : {Vd4_df.shape}")
print(f"  Vd15 : {Vd15_df.shape}")
print(f"  Vfp  : {Vfp_df.shape}")

print("\nLoading pollinator side...")
Vp_df  = pd.read_csv(VP_PATH,  index_col=0)
P_df   = pd.read_csv(P_PATH,   index_col=0)
Vpp_df = pd.read_csv(VP_PROB,  index_col=0)   # PMp
Vpt_df = pd.read_csv(VP_TEMP,  index_col=0)   # TMp
print(f"  Vp   : {Vp_df.shape}")
print(f"  Vpp  : {Vpp_df.shape}  (PMp — ANTHEIA-PMp ablation)")
print(f"  Vpt  : {Vpt_df.shape}  (TMp — ANTHEIA-TMp ablation)")

print("\nLoading curves...")
f_curves_df = pd.read_csv(F_CURVES_PATH, index_col=0)
f_curves_df.columns = list(range(52))
a_curves_df = pd.read_csv(A_CURVES_PATH, index_col=0)
a_curves_df.columns = list(range(52))
print(f"  f_curves : {f_curves_df.shape}")
print(f"  a_curves : {a_curves_df.shape}")

# Common bins
common_bins = [b for b in F_df.columns if b in set(P_df.columns)]
F_common = F_df[common_bins].values
P_common = P_df[common_bins].values
fc_idx   = {sp: i for i, sp in enumerate(F_df.index)}
pc_idx   = {sp: i for i, sp in enumerate(P_df.index)}

def compute_N(plant, pollinator):
    return float(F_common[fc_idx[plant]] @ P_common[pc_idx[pollinator]])

print(f"\nCommon bins: {len(common_bins)}")

In [ ]:
# Shared pair universe — species must be present in ALL models
plants_all = (
    set(Vf_df.index) & set(f_curves_df.index) &
    set(Vd4_df.index) & set(Vd15_df.index) & set(Vfp_df.index)
)
polls_all = set(Vp_df.index) & set(a_curves_df.index)

# For ablation models: also require PMp and TMp coverage
polls_ablation = polls_all & set(Vpp_df.index) & set(Vpt_df.index)

print(f"Plants in all primary models:    {len(plants_all)}")
print(f"Pollinators in primary models:   {len(polls_all)}")
print(f"Pollinators in ablation models:  {len(polls_ablation)}")

# Load GloBI
globi = pd.read_csv(GLOBI_PATH)
globi = globi.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species"
})
globi = globi.dropna(subset=["plant_species", "pollinator_species"])
globi = globi[["plant_species", "pollinator_species"]].drop_duplicates()

globi_shared = globi[
    globi["plant_species"].isin(plants_all) &
    globi["pollinator_species"].isin(polls_all)
].reset_index(drop=True)
print(f"Positive pairs (primary):  {len(globi_shared)}")

globi_ablation = globi[
    globi["plant_species"].isin(plants_all) &
    globi["pollinator_species"].isin(polls_ablation)
].reset_index(drop=True)
print(f"Positive pairs (ablation): {len(globi_ablation)}")

In [ ]:
# Build pair sets for primary and ablation models
def sample_negatives(positive_df, plant_pool, poll_pool, n_neg, seed):
    rng = np.random.default_rng(seed)
    pos_set = set(zip(positive_df["plant_species"], positive_df["pollinator_species"]))
    plant_list = sorted(plant_pool)
    poll_list  = sorted(poll_pool)
    negatives  = []
    while len(negatives) < n_neg:
        pl_sample = rng.choice(plant_list, size=n_neg * 2)
        po_sample = rng.choice(poll_list,  size=n_neg * 2)
        for pl, po in zip(pl_sample, po_sample):
            if (pl, po) not in pos_set:
                negatives.append((pl, po))
            if len(negatives) >= n_neg:
                break
    return pd.DataFrame(negatives, columns=["plant_species", "pollinator_species"])

# Primary pairs
neg_primary = sample_negatives(
    globi_shared, plants_all, polls_all,
    len(globi_shared) * NEG_RATIO, SEED
)
pairs_primary = pd.concat([
    globi_shared.assign(label=1),
    neg_primary.assign(label=0)
], ignore_index=True)

# Ablation pairs
neg_ablation = sample_negatives(
    globi_ablation, plants_all, polls_ablation,
    len(globi_ablation) * NEG_RATIO, SEED
)
pairs_ablation = pd.concat([
    globi_ablation.assign(label=1),
    neg_ablation.assign(label=0)
], ignore_index=True)

print(f"Primary pairs:  {len(pairs_primary)}")
print(f"Ablation pairs: {len(pairs_ablation)}")

In [ ]:
# ── PRIMARY MODELS: assemble feature matrices ─────────────────────────────────
print("Assembling primary feature matrices...")

rows_A2 = []
rows_A3 = []
rows_Ap = []
rows_As = []
rows_B  = []

for _, row in pairs_primary.iterrows():
    pl, po = row["plant_species"], row["pollinator_species"]
    vf    = Vf_df.loc[pl].values
    vp    = Vp_df.loc[po].values
    n     = compute_N(pl, po)
    f     = f_curves_df.loc[pl].values
    a     = a_curves_df.loc[po].values
    delta = np.minimum(f, a).sum()
    vd4   = Vd4_df.loc[pl].values
    vd15  = Vd15_df.loc[pl].values
    vfp   = Vfp_df.loc[pl].values

    rows_A2.append(np.concatenate([vf, vp, [n]]))          # Spatial Baseline
    rows_A3.append(np.concatenate([vf, vp, [n, delta]]))   # ANTHEIA-Scalar
    rows_Ap.append(np.concatenate([vf, vp, [n], vd4]))     # ANTHEIA-4D
    rows_As.append(np.concatenate([vf, vp, [n], vd15]))    # ANTHEIA-15D
    rows_B.append(np.concatenate([vfp, vp, [n]]))          # ANTHEIA-PMf

X_primary = {
    "Spatial Baseline" : np.array(rows_A2),
    "ANTHEIA-Scalar"   : np.array(rows_A3),
    "ANTHEIA-4D"       : np.array(rows_Ap),
    "ANTHEIA-15D"      : np.array(rows_As),
    "ANTHEIA-PMf"      : np.array(rows_B),
}
y_primary = pairs_primary["label"].values

for name, X in X_primary.items():
    print(f"  {name:<20} : {X.shape}")

In [ ]:
# ── ABLATION MODELS: assemble feature matrices ───────────────────────────────
# ANTHEIA-PMp: replace binary Vp with normalized GBIF weekly histogram (PMp)
# ANTHEIA-TMp: replace binary Vp with binary temporal existence matrix (TMp)
print("Assembling ablation feature matrices...")

rows_Bprime   = []   # ANTHEIA-PMp
rows_Bdprime  = []   # ANTHEIA-TMp

for _, row in pairs_ablation.iterrows():
    pl, po = row["plant_species"], row["pollinator_species"]
    vf   = Vf_df.loc[pl].values
    n    = compute_N(pl, po)
    vpp  = Vpp_df.loc[po].values    # PMp — GBIF normalized weekly histogram
    vpt  = Vpt_df.loc[po].values    # TMp — GBIF binary temporal existence

    rows_Bprime.append(np.concatenate([vf, vpp, [n]]))    # ANTHEIA-PMp
    rows_Bdprime.append(np.concatenate([vf, vpt, [n]]))   # ANTHEIA-TMp

X_ablation = {
    "ANTHEIA-PMp" : np.array(rows_Bprime),
    "ANTHEIA-TMp" : np.array(rows_Bdprime),
}
y_ablation = pairs_ablation["label"].values

for name, X in X_ablation.items():
    print(f"  {name:<20} : {X.shape}")

In [ ]:
# ── Train/test splits ────────────────────────────────────────────────────────
# Primary: shared split for fair comparison
idx_p = np.arange(len(pairs_primary))
train_p, test_p = train_test_split(
    idx_p, test_size=0.2, random_state=SEED, stratify=y_primary
)

# Ablation: separate split (different pair universe)
idx_a = np.arange(len(pairs_ablation))
train_a, test_a = train_test_split(
    idx_a, test_size=0.2, random_state=SEED, stratify=y_ablation
)

print(f"Primary  — train: {len(train_p)}, test: {len(test_p)}")
print(f"Ablation — train: {len(train_a)}, test: {len(test_a)}")

In [ ]:
# ── Evaluate all models ──────────────────────────────────────────────────────
def evaluate(name, X, y, train_idx, test_idx, seed=SEED):
    clf = LogisticRegression(max_iter=1000, random_state=seed)
    clf.fit(X[train_idx], y[train_idx])
    y_prob = clf.predict_proba(X[test_idx])[:, 1]
    roc = roc_auc_score(y[test_idx], y_prob)
    pr  = average_precision_score(y[test_idx], y_prob)
    return roc, pr

print("=" * 55)
print("PRIMARY MODELS (seed 42, shared pair universe)")
print("=" * 55)
print(f"{'Model':<22} {'ROC-AUC':>10} {'PR-AUC':>10} {'Dim':>5}")
print("-" * 55)
for name, X in X_primary.items():
    roc, pr = evaluate(name, X, y_primary, train_p, test_p)
    print(f"{name:<22} {roc:>10.4f} {pr:>10.4f} {X.shape[1]:>4}D")

print()
print("=" * 55)
print("ABLATION MODELS — Pollinator-Side Temporal Enrichment")
print("=" * 55)
print(f"{'Model':<22} {'ROC-AUC':>10} {'PR-AUC':>10} {'Dim':>5}")
print("-" * 55)
for name, X in X_ablation.items():
    roc, pr = evaluate(name, X, y_ablation, train_a, test_a)
    print(f"{name:<22} {roc:>10.4f} {pr:>10.4f} {X.shape[1]:>4}D")

print()
print("Note: ANTHEIA-PMp and ANTHEIA-TMp both fall below Spatial Baseline.")
print("The failure is data quality, not data quantity: GBIF pollinator")
print("observation timing reflects recorder effort, not true phenology.")
print("This asymmetry motivates the SDM comparison — see evaluation/02.")